In [1]:


from __future__ import annotations

import sys
from pathlib import Path
import numpy as np
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import tensorflow as tf
from dataclasses import dataclass
import types
from typing import Literal

# -----------------------------------------------------------------------------
# Paths: allow importing both core package and WIP package when running from repo root
# -----------------------------------------------------------------------------
def change_to_repo_root(marker: str = "src") -> None:
    """Change CWD to the repository root (parent of `src`)."""
    here = Path.cwd()
    for parent in [here] + list(here.parents):
        if (parent / marker).is_dir():
            os.chdir(parent)
            break

change_to_repo_root("WIP")
ROOT = Path.cwd()
WIP_SRC = ROOT / "WIP" / "src"
CORE_SRC = ROOT / "src"
WIP= ROOT / "WIP"
sys.path.insert(0, str(WIP_SRC))
sys.path.insert(0, str(CORE_SRC))
sys.path.insert(0, str(WIP))

from Q_Sea_Battle_New.pyr_internal_model_a import PyrInternalModelA
from Q_Sea_Battle_New.pyr_internal_model_b import PyrInternalModelB

from Q_Sea_Battle_New.pyr_measurement_layer_a import PyrMeasurementLayerA
from Q_Sea_Battle_New.pyr_combine_layer_a import PyrCombineLayerA
from Q_Sea_Battle_New.pyr_measurement_layer_b import PyrMeasurementLayerB
from Q_Sea_Battle_New.pyr_combine_layer_b import PyrCombineLayerB

# -----------------------------------------------------------------------------
# Imports (core)
# -----------------------------------------------------------------------------
from Q_Sea_Battle.game_layout import GameLayout
from Q_Sea_Battle.game_env import GameEnv
from Q_Sea_Battle.trainable_assisted_players import TrainableAssistedPlayers
from Q_Sea_Battle.tournament import Tournament
from Q_Sea_Battle.gameplay_adapters import GameplayModelAAdapter, GameplayModelBAdapter

from config import get_config
from data_build import load_dataset, build_train_pipeline


from helpers import load_ab_weights

In [2]:
config = get_config()
print(f"PR-assisted correlation parameter (P_HIGH): {float(config.get("P_HIGH", 1.0))}")
ROOT = Path(config["ROOT"])
WIP = Path(config["WIP"])
DEPTH = int(config["DEPTH"])
CHECKPOINT_DIR = Path(config["CHECKPOINT_DIR"])
MODEL_A_WEIGHTS_IN = config["MODEL_A_WEIGHTS_IN"]
MODEL_B_WEIGHTS_IN = config["MODEL_B_WEIGHTS_IN"]
FIELD_SIZE = int(config["FIELD_SIZE"])
COMMS_SIZE = int(config["COMMS_SIZE"])


PR-assisted correlation parameter (P_HIGH): 0.9


In [3]:

# -----------------------------------------------------------------------------
# Settings (as provided)
# -----------------------------------------------------------------------------

GAMES_IN_EVAL_TOURNAMENT = 1000

SEED = 1234

layout_eval = GameLayout(
    field_size=FIELD_SIZE,
    comms_size=COMMS_SIZE,
    number_of_games_in_tournament=GAMES_IN_EVAL_TOURNAMENT,
    channel_noise=0.0,
    enemy_probability=0.5,
)
depth = 4
ALPHA_FOR_SR_LAYER = 0.3 # scale this to |logit| = 10.0

## Load  models

In [4]:
def build_model_a(config: dict[str, Any]) -> tf.keras.Model:
    n2 = int(config["N2"])
    field_size = int(config.get("FIELD_SIZE", int(np.sqrt(n2))))
    comms_size = int(config.get("COMMS_SIZE", 1))
    p_high = float(config.get("P_HIGH", 1.0))
    beta_input = float(config["BETA_INPUT"])
    seed = int(config["SEED"])

    layout = GameLayout(
        field_size=field_size,
        comms_size=comms_size,
        number_of_games_in_tournament=1000,
        channel_noise=0.0,
        enemy_probability=0.5,
    )

    return PyrInternalModelA(
        layout,
        sr_mode="stochastic",
        p_high=p_high,
        beta=beta_input,
        alpha=ALPHA_FOR_SR_LAYER, # scale this to |logit| = 10.0
        seed=seed + 10,
    )



def build_model_b(config: dict[str, Any]) -> tf.keras.Model:
    n2 = int(config["N2"])
    field_size = int(config.get("FIELD_SIZE", int(np.sqrt(n2))))
    comms_size = int(config.get("COMMS_SIZE", 1))
    p_high = float(config.get("P_HIGH", 1.0))
    beta_input = float(config["BETA_INPUT"])

    layout = GameLayout(
        field_size=field_size,
        comms_size=comms_size,
        number_of_games_in_tournament=1000,
        channel_noise=0.0,
        enemy_probability=0.5,
    )

    return PyrInternalModelB(
        layout,
        sr_mode="stochastic",
        p_high=p_high,
        beta=beta_input,
        alpha=ALPHA_FOR_SR_LAYER, # scale this to |logit| = 10.0
    )



def _force_build_models(model_a: tf.keras.Model, model_b: tf.keras.Model, sample_batch: Any) -> None:
    """Warmup build via real forward calls to create variables once."""
    field_logits, gun_logits, _, meas_in_a_tgt_list, meas_out_a_tgt_list, comms_tgt_list = _unpack_batch(sample_batch)

    _ = model_a.compute_with_internal(
        field_logits=field_logits,
        replay_out_a_logits_list=meas_out_a_tgt_list,
        harden_between_levels=False,
        training=False,
    )
    if hasattr(model_a, "_ensure_built"):
        model_a._ensure_built()

    comm0 = tf.cast(comms_tgt_list[0], tf.float32)
    _ = model_b.compute_with_internal(
        gun_logits=gun_logits,
        comm_in_logits=comm0,
        prev_meas_list=meas_in_a_tgt_list,
        prev_out_list=meas_out_a_tgt_list,
        training=False,
    )
    if hasattr(model_b, "_ensure_built"):
        model_b._ensure_built()

def _unpack_batch(batch: Any) -> tuple[
        tf.Tensor,
        tf.Tensor,
        tf.Tensor,
        list[tf.Tensor],
        list[tf.Tensor],
        list[tf.Tensor],
    ]:
        field0, gun0, shoot_tgt_logits, meas_in_a_tgt_list, meas_out_a_tgt_list, comms_tgt_list = batch
        return (
            tf.cast(field0, tf.float32),
            tf.cast(gun0, tf.float32),
            tf.cast(shoot_tgt_logits, tf.float32),
            [tf.cast(x, tf.float32) for x in meas_in_a_tgt_list],
            [tf.cast(x, tf.float32) for x in meas_out_a_tgt_list],
            [tf.cast(x, tf.float32) for x in comms_tgt_list],
        )

In [5]:
raw_ds = load_dataset(config)
tfds_train = build_train_pipeline(raw_ds, config)
tfds_train = tfds_train.prefetch(tf.data.AUTOTUNE)

internal_model_a = build_model_a(config)
internal_model_b = build_model_b(config)


first_batch = next(iter(tfds_train))
_force_build_models(internal_model_a, internal_model_b, first_batch)

In [6]:

#a_in = "c:\\Users\\nly99857\\OneDrive - Philips\\SW Projects\\QSeaBattle\\WIP\\weights_pyr_models\\model_a_weights_20260225_141005.weights.h5"
#b_in = "c:\\Users\\nly99857\\OneDrive - Philips\\SW Projects\\QSeaBattle\\WIP\\weights_pyr_models\\model_b_weights_20260225_165734.weights.h5"

a_in = "C:\\Users\\nly99857\\OneDrive - Philips\\SW Projects\\QSeaBattle\\WIP\\checkpoints\\combined_ab\\model_a_step30.weights.h5"
b_in = "C:\\Users\\nly99857\\OneDrive - Philips\\SW Projects\\QSeaBattle\\WIP\\checkpoints\\combined_ab\\model_b_step30.weights.h5"


loaded = load_ab_weights(internal_model_a, internal_model_b, a_in, b_in)

[weights] skip load: missing file(s): C:\Users\nly99857\OneDrive - Philips\SW Projects\QSeaBattle\WIP\checkpoints\combined_ab\model_a_step30.weights.h5, C:\Users\nly99857\OneDrive - Philips\SW Projects\QSeaBattle\WIP\checkpoints\combined_ab\model_b_step30.weights.h5


## Play evaluation tournament

In [7]:
HARDENING_BETWEEN_LEVELS_FOR_MODEL_A = False
HARDENING_BETWEEN_LEVELS_FOR_MODEL_B = False
model_a = GameplayModelAAdapter(internal_model_a=internal_model_a, beta=10.0, harden_between_levels=HARDENING_BETWEEN_LEVELS_FOR_MODEL_A)
model_b = GameplayModelBAdapter(internal_model_b=internal_model_b, beta=10.0, harden_between_levels=HARDENING_BETWEEN_LEVELS_FOR_MODEL_B)



print("Running tournament...")
layout_eval = GameLayout(
    field_size=FIELD_SIZE,
    comms_size=COMMS_SIZE,
    number_of_games_in_tournament=GAMES_IN_EVAL_TOURNAMENT,
    channel_noise=0.0,
    enemy_probability=0.5,
)
env = GameEnv(layout_eval)
players = TrainableAssistedPlayers(layout_eval, model_a=model_a, model_b=model_b)

t = Tournament(game_env=env, players=players, game_layout=layout_eval)
log = t.tournament()

print("Tournament finished.")
mean_reward, std_err = log.outcome()
print(f"Pyramid bootstrap tournament over {layout_eval.number_of_games_in_tournament}: {mean_reward:.4f} ± {std_err:.4f}")
print(f"Weights used: model_a from {a_in}, model_b from {b_in}")
print(f"Alpha for SR layer: {ALPHA_FOR_SR_LAYER}")
print(f"PR-assisted correlation parameter (P_HIGH): {float(config.get("P_HIGH", 1.0))}")
print(f"Harden between levels: model_a={HARDENING_BETWEEN_LEVELS_FOR_MODEL_A}, model_b={HARDENING_BETWEEN_LEVELS_FOR_MODEL_B}")





Running tournament...
Tournament finished.
Pyramid bootstrap tournament over 1000: 0.4620 ± 0.0158
Weights used: model_a from C:\Users\nly99857\OneDrive - Philips\SW Projects\QSeaBattle\WIP\checkpoints\combined_ab\model_a_step30.weights.h5, model_b from C:\Users\nly99857\OneDrive - Philips\SW Projects\QSeaBattle\WIP\checkpoints\combined_ab\model_b_step30.weights.h5
Alpha for SR layer: 0.3
PR-assisted correlation parameter (P_HIGH): 0.9
Harden between levels: model_a=False, model_b=False
